# Sprint 5 — Corona Review Intelligence Chatbot

RAG-powered chatbot for analyzing Corona toilet product reviews and technical documents.

**Stack:** LangChain · ChromaDB · GPT-4o-mini · LangSmith

**Features:**
- Retrieval Augmented Generation (RAG)
- Conversation memory
- Thumbs up/down feedback logging
- LangSmith tracing
- Product Liability Radar

## 1. Setup & Imports

In [3]:
import os
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

load_dotenv()
print('Setup complete.')

Setup complete.


## 2. What is RAG?

**Retrieval Augmented Generation (RAG)** solves a key problem: LLMs don't know your private data.

Instead of fine-tuning, RAG works in 3 steps:
1. **Index** — chunk your documents and store them as vectors in ChromaDB
2. **Retrieve** — when a question comes in, find the most relevant chunks
3. **Generate** — send those chunks + the question to GPT, get a grounded answer

This means the chatbot answers from Corona's **actual review data and product docs**, not general knowledge.

## 3. Loading & Ingesting Documents

In [4]:
# Load the main review dataset
df = pd.read_csv('./context/Reviews_Reason_Classification.csv')
print(f'Total reviews: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Total reviews: 221
Columns: ['SKU', 'Product Name', 'Category', 'Author', 'Stars', 'Subject', 'Reviews', 'Review_English', 'Date', 'MarketPlace', 'Sentiment_GPT', 'YearMonth', 'Negative_Reason', 'Positive_Reason', 'Neutral_Reason']


,SKU,Product Name,Category,Author,Stars,Subject,Reviews,Review_English,Date,MarketPlace,Sentiment_GPT,YearMonth,Negative_Reason,Positive_Reason,Neutral_Reason
0,101651001,Sanitario Power One Blanco,Toilets,María Félix,5,Muy cómodo,"Cómodo, ahorrador de agua y bonito diseño. Bue...","Comfortable, water-saving, and has a nice desi...",2022-05-01,HomeCenter,Positive,2022-05,NaN,Toilet,NaN
1,101651001,Sanitario Power One Blanco,Toilets,Jorge Rhenals,5,Espectacular producto.,Compré hace unas semanas el producto y me fasc...,I bought the product a few weeks ago and I lov...,2022-07-16,HomeCenter,Positive,2022-07,NaN,General,NaN
2,101651001,Sanitario Power One Blanco,Toilets,NaN,5,Excelente,"Es un sanitario fuerte, muy econòmico en cuant...","It's a strong toilet, very economical in terms...",2022-11-12,HomeCenter,Positive,2022-11,NaN,General,NaN


In [5]:
# Quick overview of the data
print('Sentiment distribution:')
print(df['Sentiment_GPT'].value_counts())
print()
print('Marketplaces:')
print(df['MarketPlace'].value_counts())
print()
print('Unique products:', df['Product Name'].nunique())

Sentiment distribution:
Sentiment_GPT
Positive    160
Negative     37
Neutral      23
Name: count, dtype: int64

Marketplaces:
MarketPlace
HomeCenter    161
Corona         60
Name: count, dtype: int64

Unique products: 49


In [6]:
# Convert each review row into a LangChain Document
def load_csv_as_documents(filepath):
    df = pd.read_csv(filepath)
    docs = []
    for _, row in df.iterrows():
        parts = []
        for field, label in [
            ('Product Name', 'Product'), ('SKU', 'SKU'),
            ('Stars', 'Rating'), ('MarketPlace', 'Marketplace'),
            ('Sentiment_GPT', 'Sentiment'), ('Review_English', 'Review'),
            ('Negative_Reason', 'Negative Reason'), ('Positive_Reason', 'Positive Reason'),
        ]:
            if field in row and pd.notna(row[field]):
                parts.append(f'{label}: {row[field]}')
        content = '\n'.join(parts)
        if content.strip():
            docs.append(Document(
                page_content=content,
                metadata={
                    'type': 'review',
                    'product': str(row.get('Product Name', '')),
                    'sentiment': str(row.get('Sentiment_GPT', '')),
                    'marketplace': str(row.get('MarketPlace', '')),
                }
            ))
    return docs

review_docs = load_csv_as_documents('./context/Reviews_Reason_Classification.csv')
print(f'Loaded {len(review_docs)} review documents')

Loaded 221 review documents


In [7]:
# Load PDF product documents
from pypdf import PdfReader

def load_pdf(filepath):
    reader = PdfReader(filepath)
    text = ''.join([page.extract_text() or '' for page in reader.pages])
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_text(text)
    return [Document(page_content=c, metadata={'type': 'product_doc', 'source': os.path.basename(filepath)}) for c in chunks]

# Load Word docs
from docx import Document as DocxDocument

def load_docx(filepath):
    doc = DocxDocument(filepath)
    text = '\n'.join([p.text for p in doc.paragraphs if p.text.strip()])
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_text(text)
    return [Document(page_content=c, metadata={'type': 'product_doc', 'source': os.path.basename(filepath)}) for c in chunks]

print('Loaders ready.')

Loaders ready.


In [8]:
# Load all documents from context folder
all_docs = []
context_dir = './context'

for filename in os.listdir(context_dir):
    filepath = os.path.join(context_dir, filename)
    ext = filename.lower().split('.')[-1]
    print(f'Loading {filename}...')
    if ext == 'csv':
        all_docs.extend(load_csv_as_documents(filepath))
    elif ext == 'pdf':
        all_docs.extend(load_pdf(filepath))
    elif ext == 'docx':
        all_docs.extend(load_docx(filepath))

print(f'\nTotal documents indexed: {len(all_docs)}')

Loading 121611001-SANITARIO-NYREN-BCO-ficha-tecnica-comercial.pdf...
Loading Sanitario_Aluvia_Plus.docx...
Loading Sanitario_Cascade.docx...
Loading 278471001-FT-SAC-SANITARIO-ALUVIA-RD.pdf...
Loading Sanitario_Smart.docx...
Loading Negative_Reviews_Classified.csv...
Loading Reviews_Reason_Classification.csv...
Loading 121611001-SANITARIO-NYREN-BLANCO-instructivo-instalacion.pdf...
Loading Sanitario_Cima.docx...
Loading Reviews_English.csv...
Loading Reviews_GPT4o_Sentiment.csv...

Total documents indexed: 797


In [9]:
# Embed and store in ChromaDB
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    persist_directory='./chroma_db_notebook'
)
print('Vector store built!')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store built!


## 4. The RAG Chatbot with Memory

In [8]:
# Initialize LLM and memory
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=False)

def ask(question):
    # Step 1: Retrieve relevant chunks
    docs = vectorstore.similarity_search(question, k=6)
    context = '\n\n'.join([doc.page_content for doc in docs])
    
    # Step 2: Get conversation history
    history = memory.load_memory_variables({})
    chat_history = history.get('chat_history', '')
    
    # Step 3: Build prompt and call GPT
    prompt = f"""You are Maya, Corona's sharpest product intelligence analyst.
Lead with the most important insight. Be punchy, use emojis as visual markers.
Max 4-5 bullet points. End with a recommendation.

Context:
{context}

Chat History:
{chat_history}

Question: {question}
Answer:"""

    response = llm.invoke(prompt)
    answer = response.content
    
    # Step 4: Save to memory
    memory.save_context({'input': question}, {'output': answer})
    return answer

print('Chatbot ready!')

Chatbot ready!


In [9]:
# Test the chatbot
answer = ask('Which products have the most complaints?')
print(answer)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🚨 **Critical Insights on Product Complaints** 🚨

- **Recurring Issues**: The **Sanitario Ultra Ahorrador Powermax Blanco** has severe durability concerns, with **50% failure rate** reported by one customer. ⚠️
- **Service Troubles**: Customers express frustration with warranty and service responses, often attributing issues to **installation errors** rather than product flaws. 📞❌
- **Quality Control**: Other models like **Sanitario Fussion** and **Montecarlo Advance** also face complaints about defects and overall performance, indicating a trend of quality inconsistency across the brand. 🔍
- **Negative Sentiment**: The sentiment across reviews is overwhelmingly **negative**, with multiple customers expressing dissatisfaction and a lack of resolution from customer service. 😡

**Recommendation**: **Immediate action is needed** to address product quality and service response issues. Implement a thorough review of manufacturing standards and enhance customer support training to resolve com

In [10]:
# Test memory — follow-up question without repeating context
answer2 = ask('What specifically are they complaining about?')
print(answer2)

🚨 **Key Complaints Overview** 🚨

- **Incorrect Product Delivery**: Customers frequently receive the wrong items, leading to dissatisfaction and frustration. 📦❌
- **Poor Customer Service**: Many reviews highlight lack of communication and unresponsive support, leaving consumers feeling abandoned. 📞😠
- **Quality Issues**: Reports of broken or defective items upon delivery, suggesting serious quality control problems. 🔧⚠️
- **Misleading Specifications**: Some products, like toilets, do not meet advertised features (e.g., slow closure), resulting in customer disappointment. 💔🚽

**Recommendation**: **Urgently improve order fulfillment accuracy** and invest in enhancing customer service responsiveness. Additionally, conduct a comprehensive quality audit of products to ensure they meet advertised specifications and standards. This will help restore customer confidence and reduce negative feedback. 🔍👍


In [11]:
# Test with product docs
answer3 = ask('What are the specs of the Sanitario Nyren?')
print(answer3)

🚽 **Sanitario Nyren Key Specs** 🚽

- **Design**: Elegant and modern appearance that fits various bathroom styles. 🏠✨
- **Water Efficiency**: Engineered for low water consumption, promoting eco-friendly usage. 💧🌱
- **Flushing System**: Powerful flushing mechanism to ensure effective waste removal without clogging. 🔄🚿
- **Compact Size**: Ideal for smaller spaces without sacrificing comfort or performance. 📏👌
- **Durability**: Made with high-quality materials to withstand daily use and maintain appearance over time. 🛠️🔒

**Recommendation**: Highlight the eco-friendly features and compact design in marketing strategies to appeal to environmentally conscious consumers and those with limited space. Use real customer testimonials to showcase performance and satisfaction. 🌟📣


## 5. Product Liability Radar

The key business insight: not all negative reviews are the same.
- **Product defects** = Corona needs to fix the product
- **Service issues** = HomeCenter or logistics needs to fix the process

The radar separates them automatically.

In [12]:
DEFECT_KEYWORDS = [
    'design flaw', 'manufacturing', 'defective', 'broken out of box',
    'cracks', 'leaks', 'clogged', 'poor quality', 'installation impossible',
    'separated from wall', 'bad smell', 'holes', 'valve broken', 'tank cracks',
    'spare parts', 'replacement parts', 'wrong color', 'damaged'
]

SERVICE_KEYWORDS = [
    'delivery', 'never arrived', 'warranty', 'customer service',
    'no response', 'waiting', 'refund', 'sent wrong', 'installation service'
]

def run_radar():
    results = vectorstore.similarity_search(
        'defective broken poor quality design flaw clogged leaks', k=30
    )
    product_defects, service_issues = [], []
    
    for doc in results:
        text = doc.page_content.lower()
        product = doc.metadata.get('product', 'Unknown')
        if any(kw in text for kw in DEFECT_KEYWORDS):
            product_defects.append(product)
        elif any(kw in text for kw in SERVICE_KEYWORDS):
            service_issues.append(product)
    
    return product_defects, service_issues

defects, services = run_radar()
print(f'Product defects found: {len(defects)}')
print(f'Service issues found:  {len(services)}')

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Product defects found: 20
Service issues found:  2


## 6. Data Visualizations

In [10]:
# Chart 1 — Sentiment Distribution
sentiment_counts = df['Sentiment_GPT'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Count']

fig = px.pie(
    sentiment_counts,
    names='Sentiment', values='Count',
    title='Overall Sentiment Distribution',
    color='Sentiment',
    color_discrete_map={'Positive': '#2ecc71', 'Neutral': '#f1c40f', 'Negative': '#e74c3c'},
    hole=0.4,
    template='plotly_white'
)
fig.update_layout(title_x=0.5)
fig.show()

In [13]:
# Chart 2 — Defects vs Service Issues (Radar Results)
from collections import Counter

defect_counts = Counter(defects).most_common(8)
products = [x[0].replace('Sanitario ', '') for x in defect_counts]
counts = [x[1] for x in defect_counts]

fig = px.bar(
    x=counts, y=products,
    orientation='h',
    title='🚨 Most At-Risk Products (Product Defects)',
    labels={'x': 'Defect Mentions', 'y': 'Product'},
    color=counts,
    color_continuous_scale='Reds',
    template='plotly_white'
)
fig.update_layout(title_x=0.5, showlegend=False, coloraxis_showscale=False)
fig.show()

In [14]:
# Chart 3 — Defects vs Service Issues breakdown
radar_summary = pd.DataFrame({
    'Category': ['Product Defects', 'Service Issues'],
    'Count': [len(defects), len(services)],
    'Color': ['#e74c3c', '#f39c12']
})

fig = px.bar(
    radar_summary,
    x='Category', y='Count',
    title='Product Defects vs Service Issues',
    color='Category',
    color_discrete_map={'Product Defects': '#e74c3c', 'Service Issues': '#f39c12'},
    text='Count',
    template='plotly_white'
)
fig.update_layout(title_x=0.5, showlegend=False)
fig.update_traces(textposition='outside')
fig.show()

In [15]:
# Chart 4 — Sentiment by Marketplace
marketplace_sentiment = df.groupby(['MarketPlace', 'Sentiment_GPT']).size().reset_index(name='Count')

fig = px.bar(
    marketplace_sentiment,
    x='MarketPlace', y='Count',
    color='Sentiment_GPT',
    title='Sentiment by Marketplace',
    color_discrete_map={'Positive': '#2ecc71', 'Neutral': '#f1c40f', 'Negative': '#e74c3c'},
    barmode='group',
    template='plotly_white'
)
fig.update_layout(title_x=0.5)
fig.show()

In [16]:
# Chart 5 — Top 10 products by negative reviews
negative_df = df[df['Sentiment_GPT'] == 'Negative']
top_negative = negative_df['Product Name'].value_counts().head(10).reset_index()
top_negative.columns = ['Product', 'Negative Reviews']
top_negative['Product'] = top_negative['Product'].str.replace('Sanitario ', '')

fig = px.bar(
    top_negative,
    x='Negative Reviews', y='Product',
    orientation='h',
    title='Top 10 Products by Negative Reviews',
    color='Negative Reviews',
    color_continuous_scale='OrRd',
    template='plotly_white'
)
fig.update_layout(title_x=0.5, showlegend=False, coloraxis_showscale=False)
fig.show()

In [17]:
# Chart 6 — Negative reason breakdown
neg_reasons = df[df['Sentiment_GPT'] == 'Negative']['Negative_Reason'].value_counts().reset_index()
neg_reasons.columns = ['Reason', 'Count']
neg_reasons = neg_reasons[neg_reasons['Reason'].notna()]

fig = px.pie(
    neg_reasons,
    names='Reason', values='Count',
    title='Why Are Customers Unhappy? (Negative Review Reasons)',
    color_discrete_sequence=['#e74c3c', '#e67e22', '#c0392b'],
    hole=0.4,
    template='plotly_white'
)
fig.update_layout(title_x=0.5)
fig.show()

## 7. LangSmith Tracing

LangSmith tracks every call made to the LLM — inputs, outputs, latency, and user feedback.

To enable it, set these in your `.env`:
```
LANGCHAIN_API_KEY=your_key
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=corona-toilet-reviews
```

Then visit **smith.langchain.com** to see all traces in your project dashboard.

In [18]:
# Verify LangSmith connection
from langsmith import Client

try:
    client = Client()
    projects = list(client.list_projects())
    print(f'LangSmith connected! Projects: {[p.name for p in projects]}')
except Exception as e:
    print(f'LangSmith error: {e}')

LangSmith connected! Projects: ['corona-toilet-reviews']


## Summary

| Feature | Status |
|---|---|
| RAG with ChromaDB | ✅ |
| GPT-4o-mini | ✅ |
| Conversation Memory | ✅ |
| PDF + DOCX + CSV ingestion | ✅ |
| Product Liability Radar | ✅ |
| LangSmith Tracing | ✅ |
| Thumbs up/down Feedback | ✅ |
| Web UI (Flask) | ✅ |